# TMJDLC — MissAV Downloader

Fresh Colab workflow using the async MissAV API. No Playwright or Chromium is needed for the primary provider. Run cells from top to bottom.

Temporary media URLs are never printed or saved in the notebook output.

## 1. Prepare runtime

Install Git and FFmpeg only when missing, then clone or update TMJDLC. Colab runtime resets still require dependencies to be installed again.

In [ ]:
import importlib.util, shutil, subprocess, sys
from pathlib import Path

if shutil.which("git") is None:
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","git"],check=True)
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get","-qq","update"],check=True)
    subprocess.run(["apt-get","-qq","install","-y","ffmpeg"],check=True)

WORKDIR = Path("/content/tmjdlc-missav")
REPO_DIR = WORKDIR / "TMJDLC"
OUTPUT_DIR = WORKDIR / "output"
SCREENSHOT_DIR = OUTPUT_DIR / "screenshots"
WORKDIR.mkdir(parents=True,exist_ok=True)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
REPO_URL = "https://github.com/Telebotfaroff/TMJDLC.git"

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(["git","-C",str(REPO_DIR),"fetch","--depth","1","origin","main"],check=True)
    subprocess.run(["git","-C",str(REPO_DIR),"reset","--hard","FETCH_HEAD"],check=True)

%cd $REPO_DIR
print("Repository updated:",subprocess.check_output(["git","rev-parse","--short","HEAD"],text=True).strip())

## 2. Install TMJDLC and MissAV API

Editable install reads the repository dependency list. The MissAV API requires Python 3.12 or newer.

In [ ]:
import sys, subprocess, importlib.util
if sys.version_info < (3,12):
    raise RuntimeError("MissAV API requires Python 3.12 or newer.")
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."],check=True)
from tmjdlc.providers.missav import MissAVProvider
from missav_api import DownloadConfigHLS
print("TMJDLC and MissAV API are ready.")

## 3. Configure video, quality, and User-Agent

Paste the video URL and choose quality. The User-Agent is applied to the shared `curl_cffi` session used by the provider.

In [ ]:
VIDEO_URL = input("MissAV video URL: ").strip()
QUALITY = input("Quality (best/half/worst/720/1080) [best]: ").strip() or "best"
USER_AGENT = input("User-Agent [Chrome desktop]: ").strip() or (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/131.0.0.0 Safari/537.36"
)
if not VIDEO_URL.startswith(("https://missav.ws/", "https://www.missav.ws/")):
    raise ValueError("Enter a supported MissAV HTTPS video page URL.")
if QUALITY not in {"best", "half", "worst"} and (not QUALITY.isdigit() or int(QUALITY) < 1):
    raise ValueError("Invalid quality selection.")
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)
print("Configured quality:", QUALITY)
print("Custom User-Agent enabled:", bool(USER_AGENT))

## 4. Fetch video metadata (custom User-Agent)

The notebook applies your selected User-Agent to the provider's shared HTTP session before metadata or HLS requests. If Colab still receives 403, the notebook will report the failing stage; a header change cannot override an IP/network restriction.

In [ ]:
provider = MissAVProvider()
await provider.__aenter__()
# Apply the chosen User-Agent to the shared curl_cffi session used by
# metadata requests and the provider's HLS downloader.
session = provider.client.core.session
session.headers.update({"User-Agent": USER_AGENT})
print("Session User-Agent:", session.headers.get("User-Agent"))
try:
    video = await provider.get_video(VIDEO_URL)
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    print("Metadata request failed.")
    print("Exception:", type(exc).__name__)
    if status:
        print("HTTP status:", status)
    raise
metadata = provider.metadata(video)
print("Title:", metadata.get("title"))
print("Publish date:", metadata.get("publish_date"))
print("Duration:", metadata.get("length"))
print("Keywords:", metadata.get("keywords"))
print("Thumbnail:", metadata.get("thumbnail"))
if not video.m3u8_base_url:
    raise RuntimeError("The page did not expose an HLS base URL. Stop and share the metadata diagnostics.")
print("HLS source detected (URL hidden).")

## 5. Download using native MissAV HLS downloader

The same configured session is used for the provider download. The notebook only accepts newly created, non-empty video files.

In [ ]:
before_files = {p.resolve() for p in OUTPUT_DIR.rglob("*") if p.is_file()}
try:
    download_result = await provider.download(video, str(OUTPUT_DIR), quality=QUALITY)
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    print("Download request failed.")
    print("Exception:", type(exc).__name__)
    if status:
        print("HTTP status:", status)
    raise
after_files = [p for p in OUTPUT_DIR.rglob("*") if p.is_file() and p.resolve() not in before_files]
video_exts = {".mp4", ".mkv", ".ts", ".mov", ".webm"}
candidates = [p for p in after_files if p.suffix.lower() in video_exts and p.stat().st_size > 0]
if not candidates:
    raise RuntimeError(f"No new video output found. Downloader returned: {type(download_result).__name__}. Check the final downloader log above.")
video_path = max(candidates, key=lambda p: p.stat().st_mtime)
print("Download result:", type(download_result).__name__)
print("Saved:", video_path.name, "| size MiB:", round(video_path.stat().st_size / 1048576, 2))

## 6. Validate with FFprobe

A successful API return is not enough: verify that the output is a readable media file with a positive duration.

In [ ]:
probe = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration,size,format_name:stream=codec_type,codec_name,width,height","-of","json",str(video_path)],capture_output=True,text=True)
if probe.returncode:
    raise RuntimeError("FFprobe validation failed: "+probe.stderr[-1000:])
import json
probe_data = json.loads(probe.stdout)
actual_duration = float(probe_data.get("format",{}).get("duration") or 0)
print(json.dumps(probe_data,indent=2))
if actual_duration <= 0:
    raise RuntimeError("Downloaded file has no valid duration.")
print("FFprobe validation passed:",round(actual_duration,2),"seconds")

## 7. Generate screenshots

In [ ]:
timestamps = [max(1,int(actual_duration*f)) for f in (0.15,0.35,0.60,0.85)]
generated = []
for index,seconds in enumerate(timestamps,1):
    frame = SCREENSHOT_DIR/f"frame_{index}.jpg"
    result = subprocess.run(["ffmpeg","-hide_banner","-loglevel","error","-y","-ss",str(seconds),"-i",str(video_path),"-frames:v","1","-q:v","2",str(frame)],capture_output=True,text=True)
    if result.returncode == 0 and frame.exists() and frame.stat().st_size:
        generated.append(frame)
        print("Created",frame.name,"at",seconds,"seconds")
    else:
        print("Frame failed:",result.stderr[-300:])
if not generated:
    raise RuntimeError("No screenshots generated.")
from IPython.display import display, Image
for frame in generated:
    display(Image(filename=str(frame)))

## 8. Package outputs and close client

In [ ]:
try:
    await provider.__aexit__(None, None, None)
except Exception as exc:
    print("Provider cleanup warning:", type(exc).__name__)
import zipfile
from google.colab import files
archive = WORKDIR / "tmjdlc-missav-output.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(video_path, arcname=video_path.name)
    for frame in generated:
        zf.write(frame, arcname="screenshots/" + frame.name)
print("ZIP size MiB:", round(archive.stat().st_size / 1048576, 2))
files.download(str(archive))

## Troubleshooting

- If installation fails, share the complete pip error and Python version.
- If metadata fields are empty, share the page URL (not cookies or media URLs) and metadata output.
- If download fails, share the exception type and final package log lines; never share signed stream URLs or cookies.
- The API project currently requires Python 3.12+ and uses its own asynchronous HLS downloader.